# Day I18 — FOG 신호: freeze index와 에너지 threshold를 Daphnet으로 직접 계산하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I18.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I18.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 가상 신호로 본 세 상황: 걷기, 떨림, 서 있기

In [ ]:
import numpy as np

fs = 64                      # Daphnet과 같은 표본화 주파수(Hz)
win = 4 * fs                 # 4초 window = 256표본
rng = np.random.default_rng(18)
t = np.arange(win) / fs

def band_powers(x, fs):
    """한 window의 locomotor band(0.5–3 Hz)와 freeze band(3–8 Hz) power."""
    x = x - x.mean()                          # 중력(직류) 성분 제거
    p = np.abs(np.fft.rfft(x)) ** 2 / len(x)  # 256-point FFT → 주파수별 power
    f = np.fft.rfftfreq(len(x), 1 / fs)       # bin 간격 = 64/256 = 0.25 Hz
    loco = p[(f >= 0.5) & (f < 3)].sum()      # 경계 3 Hz bin은 freeze 쪽에 넣기로 정함
    frz = p[(f >= 3) & (f <= 8)].sum()
    return loco, frz

# 가상 shank 수직 가속도(mg): 세 상황
walk = 300 * np.sin(2 * np.pi * 1.0 * t) + 80 * np.sin(2 * np.pi * 2.0 * t)   # 1 Hz 걸음 + 배음
tremble = 60 * np.sin(2 * np.pi * 1.0 * t) + 150 * np.sin(2 * np.pi * 5.5 * t)  # 걸음 리듬은 약하고 5.5 Hz 떨림
stand = np.zeros(win)
for name, x in [("걷기", walk), ("제자리 떨림(FOG 흉내)", tremble), ("서 있기", stand)]:
    x = 1000 + x + rng.normal(0, 8, win)     # 중력 1 g = 1000 mg + 센서 잡음
    loco, frz = band_powers(x, fs)
    print(f"{name:14s} loco={loco:12.0f}  freeze={frz:10.0f}  합={loco + frz:12.0f}  FI={frz / loco:6.2f}")

### 예시 2 — Daphnet 데이터 사전을 코드로 확인하기

**준비:** UCI 페이지(<https://archive.ics.uci.edu/dataset/245/daphnet+freezing+of+gait>)에서 zip을 내려받아 `data/daphnet/`에 풉니다. 풀면 `data/daphnet/dataset_fog_release/dataset/S01R01.txt`처럼 됩니다. 이 레슨 작성 때 받은 zip의 SHA-256은 `27c6173b3acee75a69284903756b9011a00766c1bbb743c37fc884c5dbc39399`이었습니다(`shasum -a 256`으로 비교). `data/`는 저장소에 올리지 않습니다.

In [ ]:
import glob, os
import numpy as np
import pandas as pd

DATA = "data/daphnet/dataset_fog_release/dataset"   # UCI zip을 data/daphnet/에 푼 위치
cols = ["t_ms",
        "shank_fwd", "shank_vert", "shank_lat",       # 발목 위(shank)
        "thigh_fwd", "thigh_vert", "thigh_lat",       # 무릎 바로 위(thigh)
        "hip_fwd", "hip_vert", "hip_lat",             # 엉덩이(hip), 문서의 채널 이름은 Trunk
        "label"]                                      # 0 실험 외, 1 FOG 아님, 2 FOG

rows = []
for path in sorted(glob.glob(os.path.join(DATA, "S*R*.txt"))):
    d = pd.read_csv(path, sep=r"\s+", header=None, names=cols)
    fname = os.path.basename(path)
    lab = d["label"].to_numpy()
    onsets = np.sum((lab[1:] == 2) & (lab[:-1] != 2)) + int(lab[0] == 2)  # 라벨 2가 시작되는 지점 수
    exp = d[d["label"] > 0]
    rows.append({"subject": fname[:3], "run": fname[3:6],
                 "n": len(d), "dt_median_ms": np.median(np.diff(d["t_ms"])),
                 "label_values": sorted(d["label"].unique().tolist()),
                 "shank_vert_mean_mg": round(exp["shank_vert"].mean()),
                 "fog_episodes": onsets,
                 "fog_s": (lab == 2).sum() / 64, "nofog_s": (lab == 1).sum() / 64})   # 64 Hz → 초
runs = pd.DataFrame(rows)
print(runs.head(3).round(1).to_string(index=False))
subj = runs.groupby("subject")[["fog_episodes", "fog_s", "nofog_s"]].sum()
subj["fog_pct"] = (100 * subj["fog_s"] / (subj["fog_s"] + subj["nofog_s"])).round(1)
subj[["fog_s", "nofog_s"]] = subj[["fog_s", "nofog_s"]].round().astype(int)
print(subj.to_string())
print("파일", len(runs), "개 | 참가자", runs["subject"].nunique(), "명 | FOG 삽화 합", int(subj["fog_episodes"].sum()),
      "| 전체 길이 %.1f분" % (runs["n"].sum() / 64 / 60))
print("실험 구간 중 FOG 비율 %.1f%%" % (100 * subj["fog_s"].sum() / (subj["fog_s"].sum() + subj["nofog_s"].sum())))

### 예시 3 — 전체 데이터의 FI와 에너지, 그리고 라벨과 겹쳐 보기

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                     # 창을 띄우지 않고 파일로만 저장
import matplotlib.pyplot as plt

DATA = "data/daphnet/dataset_fog_release/dataset"
FIG = "figures/advanced/I"
os.makedirs(FIG, exist_ok=True)
fs, WIN, STEP = 64, 256, 32               # 4 s window, 0.5 s step
SHANK_VERT = 2                            # 0부터 센 열 번호: 시간(0), shank 전후(1), shank 수직(2)
POWER_TH = 1e4                            # 에너지 threshold (mg² 단위의 band power 합, 아래 본문 참고)

f = np.fft.rfftfreq(WIN, 1 / fs)
LOCO = (f >= 0.5) & (f < 3)
FRZ = (f >= 3) & (f <= 8)

def freeze_index(x, lab):
    """window 끝 시점을 '지금'으로 보고, 지난 4초로 FI와 에너지를 계산한다(causal)."""
    ends = np.arange(WIN - 1, len(x), STEP)
    seg = x[ends[:, None] - np.arange(WIN)[::-1]]          # (window 수, 256)
    seg = seg - seg.mean(axis=1, keepdims=True)
    p = np.abs(np.fft.rfft(seg, axis=1)) ** 2 / WIN
    loco, frz = p[:, LOCO].sum(1), p[:, FRZ].sum(1)
    energy = loco + frz
    fi = np.divide(frz, loco, out=np.zeros_like(frz), where=loco > 0)
    return pd.DataFrame({"t_s": ends / fs, "label": lab[ends], "energy": energy,
                         "fi": fi, "fi_gated": np.where(energy >= POWER_TH, fi, 0.0)})

def auc(score, y):
    """순위로 계산한 AUC: FOG window의 점수가 비FOG보다 높을 확률."""
    r = pd.Series(score).rank().to_numpy()
    n1, n0 = y.sum(), (~y).sum()
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

parts = []
for path in sorted(glob.glob(os.path.join(DATA, "S*R*.txt"))):
    a = np.loadtxt(path)
    w = freeze_index(a[:, SHANK_VERT], a[:, 10])
    w["subject"], w["run"] = os.path.basename(path)[:3], os.path.basename(path)[3:6]
    parts.append(w)
win = pd.concat(parts, ignore_index=True)
win = win[win["label"] > 0].copy()                    # 라벨 0(실험 외) window는 뺀다
win["fog"] = win["label"] == 2

rows = []
for s, g in win.groupby("subject"):
    y = g["fog"].to_numpy()
    row = {"subject": s, "windows": len(g), "fog_windows": int(y.sum()),
           "low_energy_%_nonFOG": round(100 * (g.loc[~g.fog, "energy"] < POWER_TH).mean(), 1)}
    if y.any():
        row["FI_median_FOG"] = round(g.loc[g.fog, "fi"].median(), 2)
        row["FI_median_nonFOG"] = round(g.loc[~g.fog, "fi"].median(), 2)
        row["AUC_FI"] = round(auc(g["fi"].to_numpy(), y), 3)
        row["AUC_FI_gated"] = round(auc(g["fi_gated"].to_numpy(), y), 3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

y = win["fog"].to_numpy()
print("\n전체 window %d개 (FOG %d개, %.1f%%)" % (len(win), y.sum(), 100 * y.mean()))
print("에너지 threshold 아래 비율: 비FOG %.1f%% | FOG %.1f%%" % (
    100 * (win.loc[~win.fog, "energy"] < POWER_TH).mean(), 100 * (win.loc[win.fog, "energy"] < POWER_TH).mean()))
for th in [1.5, 2.0, 3.0]:
    pred = win["fi_gated"].to_numpy() > th
    print("FreezeTH %.1f: 민감도 %.1f%%, 특이도 %.1f%% (전체 window를 한데 모은 값)" % (
        th, 100 * pred[y].mean(), 100 * (~pred[~y]).mean()))

# 그림: 한 run에서 FI·에너지와 FOG 라벨 겹쳐 보기
g = win[(win.subject == "S02") & (win.run == "R01")]
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for a_ in ax:
    on = g["fog"].to_numpy()
    a_.fill_between(g["t_s"], 0, 1, where=on, transform=a_.get_xaxis_transform(),
                    color="tab:red", alpha=0.2, step="post", label="FOG label")
ax[0].plot(g["t_s"], g["fi"], lw=0.8, color="0.3", label="FI")
ax[0].set_yscale("log"); ax[0].set_ylabel("freeze index"); ax[0].legend(loc="upper right")
ax[1].plot(g["t_s"], g["energy"], lw=0.8, color="tab:blue")
ax[1].axhline(POWER_TH, ls="--", color="k", lw=0.8)
ax[1].set_yscale("log"); ax[1].set_ylabel("band energy (mg²)"); ax[1].set_xlabel("time (s)")
fig.suptitle("Daphnet S02R01, shank vertical, 4 s window / 0.5 s step")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "I18_fi_overlay.png"), dpi=120)
print("저장:", os.path.join(FIG, "I18_fi_overlay.png"))